# Day 6 – Unsupervised Learning



---

## 1. Learning Objectives

- Explain the difference between supervised and unsupervised learning.
- Apply K-Means clustering and interpret cluster profiles.
- Use the Elbow Method and Silhouette Score to choose optimal K.
- Apply Hierarchical Clustering and read a dendrogram.
- Reduce dataset dimensions with PCA for 2D visualization.
- Implement a simple reinforcement learning bandit simulation.

## Concept / Theory

### Unsupervised vs Supervised
| Aspect | Supervised | Unsupervised |
|---|---|---|
| Labels | Required | Not required |
| Goal | Predict output | Discover structure |
| Examples | Classification, Regression | Clustering, PCA |

### K-Means Algorithm
1. Initialize K centroids randomly
2. Assign each point to nearest centroid
3. Move centroids to cluster means
4. Repeat 2-3 until centroids don't change

### PCA (Principal Component Analysis)
Projects high-dimensional data onto fewer dimensions while preserving maximum variance. Used for visualization and noise reduction.

### Reinforcement Learning (Intro)
An agent takes actions in an environment, receives rewards, and learns to maximize cumulative reward over time.

In [ ]:
# Setup
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.datasets import load_iris
from scipy.cluster.hierarchy import dendrogram, linkage
print('Ready!')

##  Practical Lab 1 – Customer Segmentation with K-Means

### Objective
Segment customers by income and spending score.

### Explanation
K-Means groups similar customers together. Businesses use this to target marketing campaigns.

In [ ]:
# Lab 1: Generate customer data
np.random.seed(42)
g1 = np.random.randn(60, 2) + [2, 7]    # Low income, High spending
g2 = np.random.randn(60, 2) + [7, 3]    # High income, Low spending
g3 = np.random.randn(60, 2) + [8, 8]    # High income, High spending
g4 = np.random.randn(30, 2) + [3, 2]    # Low income, Low spending

X_cust = np.vstack([g1, g2, g3, g4])
df_c = pd.DataFrame(X_cust, columns=['Income', 'Spending_Score'])

plt.figure(figsize=(7, 5))
plt.scatter(df_c['Income'], df_c['Spending_Score'], alpha=0.6, c='steelblue')
plt.xlabel('Annual Income ($k)'); plt.ylabel('Spending Score')
plt.title('Customer Data (Unlabeled)')
plt.show()
print(f'Dataset: {df_c.shape[0]} customers')

In [ ]:
# Apply K-Means (K=4)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_cust)

km = KMeans(n_clusters=4, n_init=15, random_state=42)
labels = km.fit_predict(X_sc)
df_c['Cluster'] = labels

# Visualize
palette = ['red', 'green', 'blue', 'orange']
plt.figure(figsize=(8, 6))
for k in range(4):
    mask = labels == k
    plt.scatter(X_cust[mask, 0], X_cust[mask, 1],
                c=palette[k], label=f'Cluster {k}', alpha=0.7, s=40)
plt.xlabel('Income'); plt.ylabel('Spending Score')
plt.title('K-Means Customer Segments (K=4)')
plt.legend(); plt.show()

print('Cluster profiles:')
print(df_c.groupby('Cluster').agg({'Income':'mean','Spending_Score':'mean'}).round(2))

### Expected Output
A scatter plot with 4 distinct colored clusters. Cluster profiles show different income/spending combinations.

##  Practical Lab 2 – Elbow Method & Silhouette Analysis

### Objective
Choose the optimal number of clusters using two methods.

### Explanation
- **Elbow Method:** Plot inertia vs K. The 'elbow' point is where adding more clusters gives diminishing returns.
- **Silhouette Score:** Ranges [-1, 1]. Higher is better. Measures how similar a point is to its own cluster vs other clusters.

In [ ]:
# Lab 2: Elbow + Silhouette
inertias   = []
silhouettes= []
K_range = range(2, 10)
for k in K_range:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_k = km_k.fit_predict(X_sc)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(X_sc, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(K_range, inertias, 'bo-', lw=2)
axes[0].set_xlabel('K (Number of Clusters)'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method'); axes[0].axvline(x=4, color='red', linestyle='--', label='Optimal K=4')
axes[0].legend()
axes[1].plot(K_range, silhouettes, 'rs-', lw=2)
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
for k, s in zip(K_range, silhouettes):
    axes[1].annotate(f'{s:.2f}', (k, s), textcoords='offset points', xytext=(0,8), fontsize=8)
plt.tight_layout(); plt.show()
best_k = list(K_range)[silhouettes.index(max(silhouettes))]
print(f'Best K by Silhouette: {best_k}  Score: {max(silhouettes):.4f}')

##  Practical Lab 3 – Hierarchical Clustering & Dendrogram

### Objective
Cluster data without specifying K upfront, then read the dendrogram.

### Explanation
Hierarchical clustering builds a tree of clusters (dendrogram). You cut the tree at a desired number of clusters after inspecting it.

In [ ]:
# Lab 3: Hierarchical clustering
# Use Iris for a well-known example
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data[:40])  # first 40 samples

# Compute linkage matrix
Z = linkage(X_iris, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z, labels=iris.target[:40],
           color_threshold=4, leaf_rotation=90, leaf_font_size=9)
plt.title('Hierarchical Clustering Dendrogram (Iris first 40 samples)')
plt.xlabel('Sample index (color = true class)')
plt.ylabel('Distance')
plt.axhline(y=4, color='red', linestyle='--', label='Cut threshold')
plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Apply agglomerative clustering
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
hier_labels = agg.fit_predict(X_iris)

# Compare with true labels
print('True labels (first 40):', iris.target[:40].tolist())
print('Hier labels:           ', hier_labels.tolist())

plt.figure(figsize=(7,5))
for k in range(3):
    mask = hier_labels == k
    plt.scatter(X_iris[mask,0], X_iris[mask,1], label=f'Cluster {k}', alpha=0.8)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Hierarchical Clustering Result')
plt.legend(); plt.show()

## Practical Lab 4 – PCA for Dimensionality Reduction

### Objective
Reduce the 4D Iris dataset to 2D for visualization.

### Explanation
**PCA** finds the directions (principal components) of maximum variance. Projecting onto the first 2 PCs lets us visualize high-dimensional data.

In [ ]:
# Lab 4: PCA on full Iris dataset
X_iris_full = StandardScaler().fit_transform(iris.data)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris_full)

print('Variance explained by each component:')
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {ev:.4f} ({ev*100:.1f}%)')
print(f'Total: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.1f}%)')

# Visualize
colors = ['red','green','blue']
plt.figure(figsize=(8,6))
for i, species in enumerate(iris.target_names):
    mask = iris.target == i
    plt.scatter(X_pca[mask,0], X_pca[mask,1], c=colors[i], label=species, s=50, alpha=0.8)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA: Iris dataset projected to 2D')
plt.legend(); plt.show()

In [ ]:
# Scree plot
pca_full = PCA(n_components=4).fit(X_iris_full)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
plt.figure(figsize=(7, 4))
plt.bar(range(1, 5), pca_full.explained_variance_ratio_, alpha=0.7, label='Individual')
plt.step(range(1, 5), cumulative, where='mid', color='red', lw=2, label='Cumulative')
plt.xlabel('Principal Component'); plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot — Iris')
plt.legend(); plt.tight_layout(); plt.show()
print('Cumulative variance:', cumulative.round(4))

### Expected Output
```
PC1: ~73%  PC2: ~23%  Total: ~96%
The 2D plot clearly separates Setosa from others.
```

## Practical Lab 5 – Reinforcement Learning: Multi-Armed Bandit

### Objective
Simulate a simple RL agent that learns which action gives the best reward.

### Explanation
The Multi-Armed Bandit problem: an agent chooses one of K actions at each step. Each action has an unknown reward distribution. The agent must balance **exploration** (try new actions) and **exploitation** (use the best known action). `epsilon-greedy` strategy: with probability ε, explore randomly; otherwise exploit the best known action.

In [ ]:
# Lab 5: Epsilon-Greedy Multi-Armed Bandit
np.random.seed(42)

# 4 actions (slot machines) with different true reward means
true_means = [0.3, 0.7, 0.5, 0.9]   # Best action is 3 (mean=0.9)
n_actions   = len(true_means)
steps       = 500
epsilon     = 0.15

# Agent state
value_estimates = np.zeros(n_actions)
action_counts   = np.zeros(n_actions)
reward_history  = []
action_history  = []

for step in range(steps):
    # Choose action: explore vs exploit
    if np.random.rand() < epsilon:
        action = np.random.randint(n_actions)   # explore
    else:
        action = np.argmax(value_estimates)     # exploit
    
    # Get reward (noisy)
    reward = np.random.normal(true_means[action], 0.1)
    
    # Update estimate (incremental average)
    action_counts[action] += 1
    value_estimates[action] += (reward - value_estimates[action]) / action_counts[action]
    
    reward_history.append(reward)
    action_history.append(action)

# Results
print('Learned value estimates:', value_estimates.round(3))
print('True means:             ', true_means)
print('Action counts:          ', action_counts.astype(int))
print(f'\nBest learned action: {np.argmax(value_estimates)} (true best: {np.argmax(true_means)})')

In [ ]:
# Visualize learning curve
cumulative_avg = np.cumsum(reward_history) / np.arange(1, steps+1)
plt.figure(figsize=(10, 5))
plt.plot(cumulative_avg, lw=2, color='blue', label='Agent average reward')
plt.axhline(y=max(true_means), color='red', lw=1.5, linestyle='--', label=f'Optimal reward ({max(true_means)})')
plt.xlabel('Step'); plt.ylabel('Average Reward')
plt.title(f'Epsilon-Greedy Bandit (ε={epsilon})')
plt.legend(); plt.ylim(0, 1.1); plt.tight_layout(); plt.show()

##  Exercises

**Exercise 1:** Run K-Means on the customer data with K=2, 3, 4, 5. Which K do you prefer and why?

In [ ]:
# Exercise 1 — Compare different K values

**Exercise 2:** Apply PCA to the Breast Cancer dataset (30 features). How many components are needed to explain 95% of variance?

In [ ]:
# Exercise 2
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
cancer = load_breast_cancer()
X = StandardScaler().fit_transform(cancer.data)
# Find components for 95% variance

**Exercise 3:** Change epsilon to 0.01 and 0.5 in the bandit simulation. How does this affect convergence?

In [ ]:
# Exercise 3 — Re-run bandit with epsilon=0.01 and epsilon=0.5, compare plots

##  Challenge Task – Mall Customer Segmentation

Create a synthetic mall customer dataset with: CustomerID, Gender (M/F), Age, Annual_Income, Spending_Score (1-100). Cluster customers into 5 segments and name each segment meaningfully.

In [ ]:
# Challenge Task: Mall Customer Segmentation
np.random.seed(7)
n = 200
df_mall = pd.DataFrame({
    'CustomerID':   range(1, n+1),
    'Gender':       np.random.choice(['M','F'], n),
    'Age':          np.random.randint(18, 70, n),
    'Annual_Income':np.random.choice(
        [np.random.normal(m, 5) for m in [25,50,75,55,90] for _ in range(n//5)], n),
    'Spending_Score':np.random.choice(
        [np.random.normal(m, 8) for m in [80,40,45,75,20] for _ in range(n//5)], n)
})
df_mall['Annual_Income']   = df_mall['Annual_Income'].clip(15, 120)
df_mall['Spending_Score']  = df_mall['Spending_Score'].clip(1, 100)

X_mall = df_mall[['Annual_Income','Spending_Score']]
X_mall_sc = StandardScaler().fit_transform(X_mall)
km_mall = KMeans(n_clusters=5, n_init=15, random_state=7)
df_mall['Segment'] = km_mall.fit_predict(X_mall_sc)

segment_names = {}
for seg in range(5):
    sub = df_mall[df_mall['Segment']==seg]
    income = sub['Annual_Income'].mean()
    spending = sub['Spending_Score'].mean()
    if income > 65 and spending > 60: name = 'Premium'
    elif income > 65 and spending < 40: name = 'Savers'
    elif income < 40 and spending > 60: name = 'Spenders'
    elif income < 40 and spending < 40: name = 'Budget'
    else: name = 'Average'
    segment_names[seg] = name
df_mall['Segment_Name'] = df_mall['Segment'].map(segment_names)

print('Segment profiles:')
print(df_mall.groupby('Segment_Name')[['Annual_Income','Spending_Score','Age']].mean().round(1))

plt.figure(figsize=(8,6))
for seg_name in df_mall['Segment_Name'].unique():
    sub = df_mall[df_mall['Segment_Name']==seg_name]
    plt.scatter(sub['Annual_Income'], sub['Spending_Score'], label=seg_name, alpha=0.7)
plt.xlabel('Annual Income ($k)'); plt.ylabel('Spending Score')
plt.title('Mall Customer Segments'); plt.legend(); plt.show()

## Real-World AI/ML Application

| Algorithm | Use Case |
|---|---|
| **K-Means** | Customer segmentation, image compression, document clustering |
| **Hierarchical** | Gene expression analysis, language family trees |
| **PCA** | Facial recognition preprocessing, noise reduction, visualization |
| **Bandit / RL** | Ad recommendation, A/B testing optimization, game AI |

## Day Summary

| Concept | Key Point |
|---|---|
| K-Means | Partition data into K clusters by minimizing inertia |
| Elbow Method | Plot inertia vs K; look for the bend |
| Silhouette Score | [-1,1]; higher = better cluster separation |
| Hierarchical | Dendrogram; choose K by cutting the tree |
| PCA | Project high-D data to fewer dimensions |
| Bandit / RL | Explore-exploit trade-off; update value estimates |

**→ Day 7: Neural Network Fundamentals**